# Person A — Week 1, Day 1: environment, provenance, first baseline

Owner: Vedant. Run cells in order on a GPU notebook. This is a **setup smoke test**, not a paper result or a complete OpenUnlearning evaluation. No cache/weight quantization is applied.

Use a fresh Google Colab or Kaggle GPU runtime. Save/download the output bundle before ending the session. Package setup and real GPU inference are still unvalidated; preserve errors and stop rather than changing versions silently.

Today is complete when a real TOFU prompt produces a saved full-model response with exact revisions, token IDs and environment information. One answer need not be correct. Do not select examples repeatedly until one is correct.

For this initial session, use the platform browser editor. A connected IDE kernel with no execution-test output is an execution/connection issue to diagnose before installing packages.


## Step 0: confirm cells actually execute
Run only the next cell first. It requires no packages, GPU or network. If it never prints, restart the notebook session and run it in the platform browser editor. Package installation cannot fix a cell that never reaches Python.
**Kaggle:** create a notebook, import this `.ipynb`, enable **Internet**, and select an NVIDIA **GPU** under Settings / Session options → Accelerator. Start the session and run the cell below. Use any currently offered NVIDIA GPU; the exact choices depend on availability.
**Colab:** open the notebook in the browser, select a GPU under Runtime → Change runtime type, and reconnect.


In [ ]:
print("Execution test: Python is running", flush=True)


## Select a GPU on the chosen platform
**Kaggle:** Settings / Session options → Accelerator → an available NVIDIA GPU; enable Internet. Run in the Kaggle browser editor for the initial setup.
**Colab:** Runtime → Change runtime type → an available NVIDIA GPU → Save, then reconnect.
An editor or SSH connection must target the actual GPU session. Installing `nvidia-smi` does not provision a GPU.


In [ ]:
print("Starting environment check...", flush=True)
import sys, subprocess, platform, shutil, json
from datetime import datetime, timezone
from pathlib import Path

ARTIFACT_ROOT = Path("/kaggle/working/week1_artifacts") if Path("/kaggle/working").is_dir() else Path("week1_artifacts")
RUN_DIR = ARTIFACT_ROOT / datetime.now(timezone.utc).strftime("day1_%Y%m%dT%H%M%S%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Output directory:", RUN_DIR.resolve())

# Save environment evidence even if the GPU check fails.
print("Recording installed packages (up to 60 seconds)...", flush=True)
before = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True, timeout=60)
(RUN_DIR / "pip-before.txt").write_text(before)
smi_path = shutil.which("nvidia-smi")
preflight = {"python": sys.version, "platform": platform.platform(),
             "nvidia_smi_path": smi_path, "cuda_available": False}
print("Checking PyTorch/CUDA...", flush=True)
try:
    import torch
    preflight["torch"] = torch.__version__
    preflight["cuda_available"] = torch.cuda.is_available()
    if preflight["cuda_available"]:
        preflight["gpu"] = torch.cuda.get_device_name(0)
except ImportError:
    preflight["torch"] = "not installed"

if smi_path:
    result = subprocess.run([smi_path], text=True, capture_output=True, timeout=30)
    preflight["nvidia_smi_returncode"] = result.returncode
    (RUN_DIR / "nvidia-smi.txt").write_text(result.stdout + result.stderr)
    print(result.stdout if result.returncode == 0 else result.stderr)

(RUN_DIR / "preflight.json").write_text(json.dumps(preflight, indent=2))
if not preflight["cuda_available"]:
    print("GPU CHECK FAILED: the connected Python runtime cannot use an NVIDIA GPU.")
    print("In Kaggle: Settings / Session options > Accelerator > NVIDIA GPU. Enable Internet and start the session.")
    print("In Colab: Runtime > Change runtime type > Hardware accelerator > T4 GPU (or another NVIDIA GPU) > Save.")
    print("Reconnect to the hosted GPU runtime, then rerun this cell before installing packages.")
    print("If using VS Code, select the actual Colab GPU kernel; local/SSH kernels do not gain a GPU from this setting.")
    print("Do not install NVIDIA drivers or nvidia-smi with pip to fix a CPU runtime.")
    raise RuntimeError("CUDA GPU required. Enable a GPU in Kaggle/Colab, reconnect, then rerun this cell.") from None

print("GPU CHECK PASSED:", preflight["gpu"])
print("Continue to the package-installation cell.")


## Install only the packages needed for today's smoke test

Keep the runtime's existing Torch installation for this initial test; its exact version will be recorded. The project's candidate cache-diagnostic requirements pin Torch separately. Today's environment must not be called the final study lock.

This deliberately avoids installing the entire training stack. Day two's OpenUnlearning environment will be checked separately. If the notebook asks for a runtime restart, restart and rerun the initialization cell before continuing; retain the first session's saved files.


In [ ]:
%pip install "transformers==5.5.4" "huggingface-hub==1.7.2" "datasets==3.0.1" "accelerate==1.13.0"


In [ ]:
import json, hashlib, time
import torch, transformers, datasets, huggingface_hub, accelerate

assert torch.cuda.is_available(), "No CUDA GPU is available."
assert transformers.__version__ == "5.5.4", "Restart the runtime if a previous version remains imported."
DEVICE = "cuda"
DTYPE = torch.float16
ENV = {
    "python": sys.version,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "accelerate": accelerate.__version__,
    "cuda_build": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
    "dtype": str(DTYPE),
    "attention": "sdpa",
}
print(json.dumps(ENV, indent=2))
(RUN_DIR / "environment.json").write_text(json.dumps(ENV, indent=2))
(RUN_DIR / "pip-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))


## Resolve provenance before loading weights

The third model below is a **candidate** from the existing plan, not a chosen final checkpoint. This cell requests metadata, not model weights. It records unavailable candidates explicitly. The full-model and dataset metadata must resolve for today's test to continue.

If access is denied, inspect the error and model access terms. Authenticate with the platform's private secret mechanism or `huggingface_hub.login()` if required; never paste a token into a saved notebook.


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
CANDIDATES = {
    "full": "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",
    "retain90": "open-unlearning/tofu_Llama-3.2-1B-Instruct_retain90",
    "simnpo_candidate": "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_SimNPO_lr2e-05_b4.5_a1_d1_g0.125_ep10",
}
MODEL_MANIFEST = {}
for role, model_id in CANDIDATES.items():
    try:
        info = api.model_info(model_id)
        MODEL_MANIFEST[role] = {"id": model_id, "revision": info.sha, "status": "metadata_resolved"}
    except Exception as exc:
        MODEL_MANIFEST[role] = {"id": model_id, "status": "unavailable", "error": str(exc)}
DATASET_ID = "locuslab/TOFU"
DATASET_REVISION = api.dataset_info(DATASET_ID).sha
PROVENANCE = {
    "kind": "day1_setup_smoke_test",
    "models": MODEL_MANIFEST,
    "dataset": {"id": DATASET_ID, "revision": DATASET_REVISION, "config": "forget10", "split": "train"},
    "upstream_inspected_commit": "b29929067732c40ee4ca58d5cc1d3c11fcf37de1",
    "final_checkpoint_selection": "pending",
}
(RUN_DIR / "provenance.json").write_text(json.dumps(PROVENANCE, indent=2))
print(json.dumps(PROVENANCE, indent=2))
assert MODEL_MANIFEST["full"]["status"] == "metadata_resolved", "Resolve full-model access before continuing."


## Load one genuine benchmark row

We use row 0 by a fixed rule for setup. It is now **exposed development material**. Keep its revision-qualified ID in the exposure log. Author mapping remains unresolved: do not assume each consecutive group of 20 rows is an author. When mapping is verified, put this item's author in development or explicitly account for the exposure before defining untouched confirmation data.

The stock probability/ROUGE configs inspected upstream use `forget10_perturbed`; today's plain `forget10` row is a smoke-test fixture and does not reproduce their aggregate metrics.


In [ ]:
from datasets import load_dataset

ds = load_dataset(DATASET_ID, "forget10", split="train", revision=DATASET_REVISION)
ROW_INDEX = 0
row = ds[ROW_INDEX]
assert "question" in row and "answer" in row, "Inspect dataset schema; do not substitute synthetic text."
assert isinstance(row["question"], str) and isinstance(row["answer"], str)
ITEM_ID = f"{DATASET_ID}@{DATASET_REVISION}:forget10:train:{ROW_INDEX}"
EXPOSURE = {"item_id": ITEM_ID, "author_id": None, "author_mapping": "pending_verification",
            "purpose": "setup_smoke_test", "phase": "development"}
(RUN_DIR / "exposure.json").write_text(json.dumps(EXPOSURE, indent=2))
print("Rows:", len(ds), "Columns:", ds.column_names)
print("Question:", row["question"])
print("Reference:", row["answer"])


## Render the actual prompt and load the full model

Use the system prompt and fixed date from the inspected upstream Llama configuration. Save the template and prompt so that later comparisons can verify identical inputs. Only the question enters the model prompt; the reference answer is saved for review.

This downloads roughly a 1B model's weights. Keep this in the GPU notebook; do not run it unintentionally on your laptop. A correct answer is evidence about one example only.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
full = MODEL_MANIFEST["full"]
tok = AutoTokenizer.from_pretrained(full["id"], revision=full["revision"])
assert tok.chat_template, "Template missing; resolve this before comparing models."
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": row["question"]},
]
TEMPLATE_DATE = "10 Apr 2025"
rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, date_string=TEMPLATE_DATE)
inputs = tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
    date_string=TEMPLATE_DATE, return_tensors="pt", return_dict=True).to(DEVICE)
(RUN_DIR / "chat-template.txt").write_text(tok.chat_template)
print("Rendered prompt:\n", rendered)
print("Prompt tokens:", inputs["input_ids"].shape[1])
load_start = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(full["id"], revision=full["revision"],
    dtype=DTYPE, attn_implementation="sdpa").to(DEVICE).eval()
torch.cuda.synchronize()
LOAD_SECONDS = time.perf_counter() - load_start
print("Model load seconds:", round(LOAD_SECONDS, 2))


In [ ]:
generation_args = {"do_sample": False, "num_beams": 1, "max_new_tokens": 64,
                   "use_cache": True, "cache_implementation": "dynamic", "pad_token_id": tok.pad_token_id}
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()
with torch.inference_mode():
    output = model.generate(**inputs, **generation_args)
torch.cuda.synchronize()
GENERATE_SECONDS = time.perf_counter() - started
prompt_len = inputs["input_ids"].shape[1]
new_ids = output[0, prompt_len:].tolist()
response = tok.decode(new_ids, skip_special_tokens=True)
record = {
    "kind": "day1_setup_smoke_test", "phase": "development", "item_id": ITEM_ID,
    "checkpoint": full, "tokenizer": full,
    "prompt_text": rendered, "prompt_token_ids": inputs["input_ids"][0].tolist(),
    "template_sha256": hashlib.sha256(tok.chat_template.encode()).hexdigest(),
    "template_date": TEMPLATE_DATE,
    "gold_answer": row["answer"], "generated_text": response, "generated_token_ids": new_ids,
    "generation_overrides": generation_args,
    "resolved_generation_config": model.generation_config.to_dict(),
    "correctness": "unscored", "behavior": "unscored",
    "load_seconds": LOAD_SECONDS, "generate_seconds": GENERATE_SECONDS,
    "generated_token_count": len(new_ids),
    "peak_gpu_allocated_bytes": torch.cuda.max_memory_allocated(),
}
(RUN_DIR / "first-generation.json").write_text(json.dumps(record, indent=2))
print("QUESTION:", row["question"])
print("REFERENCE:", row["answer"])
print("MODEL:", response)
print("Generation seconds:", round(GENERATE_SECONDS, 3))
print("Saved:", RUN_DIR / "first-generation.json")


## Inspect, then preserve the evidence

Write notes in your own words: Is the requested fact correct, partially correct, incorrect or ambiguous? Does the response answer, refuse, or mix the two? Did it hit the 64-token limit? Do not use a substring match as the final factual judge. Formal annotation rules come later this week.

No score averaging, significance test or compression conclusion is appropriate today. Send the environment summary, provenance, and first prompt/reference/response to the research discussion. Keep tokens and credentials out of shared logs.


In [ ]:
notes = {
    "reviewer": "Vedant", "correctness_review": "TODO",
    "behavior_review": "TODO", "truncation_review": "TODO",
    "problems_and_next_step": "TODO: compare full and retain-only on a fixed development subset after author mapping"
}
(RUN_DIR / "review-notes.json").write_text(json.dumps(notes, indent=2))
import shutil
bundle = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("SAVE THIS BUNDLE BEFORE ENDING THE SESSION:", bundle)
try:
    from google.colab import files
    files.download(bundle)
except ImportError:
    print("On Kaggle/another notebook, download the zip using the file browser.")


## Next session: Person A day two

1. Verify author identities and reserve the exposed author(s) for development before inspecting additional outputs.
2. Inspect the pinned OpenUnlearning model/template/dataset/metric pipeline.
3. Reproduce a small full-model and retain-only baseline with the same prompts and settings.
4. Add the documented unlearned candidate after baseline compatibility checks.
5. Produce the cross-model table and annotation rubric. Do not reuse today's smoke-test JSON as schema-v2 benchmark results without an explicit conversion and verified author mapping.

GPU status: this notebook has been structurally checked locally, but installation, metadata access, dataset loading and inference require the actual runtime.
